# Home Credit modeling: leakage-safe experiments and final submission

This notebook compares models for predicting `TARGET`, tunes the leading candidate, and fits the retained configuration for Kaggle submission. It preserves the majority-class and weighted/unweighted logistic-regression baselines, bounded random forest, and weighted/unweighted histogram gradient boosting. Model selection uses ROC-AUC; accuracy is retained to demonstrate why it is misleading for an imbalanced target.

## Table of contents

1. [Introduction](#Introduction)
2. [Data preparation summary](#Data-preparation-summary)
3. [Modeling process](#Modeling-process)
4. [Performance comparisons](#Performance-comparisons)
5. [Hyperparameter tuning](#Hyperparameter-tuning)
6. [Final model selection](#Final-model-selection)
7. [Kaggle results](#Kaggle-results)
8. [Final limitations](#Final-limitations)

## Introduction

The objective is to predict Home Credit's binary `TARGET` while maximizing ROC-AUC. Because only about 8% of labeled applicants are positive, accuracy is reported for context but ROC-AUC governs model comparison and selection. The notebook progresses from simple baselines through bounded tree candidates and sample-based tuning to a reproducible final fit and authenticated Kaggle result.

### Experiment contract

- Use the raw application training table so every learned preparation parameter can be fitted inside cross-validation.
- Use the same shuffled, 3-fold stratified splits for every model (`random_state=42`).
- Exclude `SK_ID_CURR` from model features and retain it only as an identifier.
- Fit preparation thresholds, the missingness schema, median/mode imputers, one-hot categories, and scaling on each training fold only.
- Compare identical logistic-regression specifications with no class weighting and with balanced class weights.
- Evaluate bounded tree candidates before tuning; defer blending, SHAP, bootstrap intervals, and subgroup analysis until the required fit and submission work is complete.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import os
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display

os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    DEFAULT_N_SPLITS,
    DEFAULT_RANDOM_STATE,
    build_baseline_models,
    build_candidate_models,
    build_tuned_hist_gradient_boosting_pipeline,
    compare_paired_auc_results,
    create_kaggle_submission,
    evaluate_model_cv,
    fit_final_hist_gradient_boosting,
    histogram_boosting_search_space,
    load_saved_pipeline,
    load_test_data,
    load_training_data,
    make_stratified_splits,
    save_pipeline_and_configuration,
    tune_hist_gradient_boosting,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 6)

environment = pd.Series({
    'Python': platform.python_version(),
    'pandas': pd.__version__,
    'NumPy': np.__version__,
    'scikit-learn': sklearn.__version__,
    'CV folds': DEFAULT_N_SPLITS,
    'random_state': DEFAULT_RANDOM_STATE,
}, name='value')
environment.to_frame()

,value
Python,3.13.9
pandas,2.3.3
NumPy,2.3.5
scikit-learn,1.7.2
CV folds,3
random_state,42


## Data preparation summary

The completed application-level feature pipeline adds missingness and data-quality flags, training-fitted extreme-value indicators, stable monetary transforms, applicant age and employment features, and affordability ratios. Raw application tables enter the modeling pipeline so every learned threshold, missingness schema, imputation value, and category vocabulary can be fitted at the correct scope.

### Load and validate the modeling population

The competition test table has no `TARGET`, so its AUC cannot be measured locally. Cross-validated out-of-fold AUC is the model-development estimate; the authenticated public and private Kaggle AUCs are recorded in the final section.

In [2]:
TRAIN_PATH = PROJECT_ROOT / 'data/raw/application_train.csv'
X, y = load_training_data(TRAIN_PATH)
cv_splits = make_stratified_splits(
    X, y, n_splits=DEFAULT_N_SPLITS, random_state=DEFAULT_RANDOM_STATE
)

dataset_summary = pd.Series({
    'rows': len(X),
    'raw candidate features (including ID before removal)': X.shape[1],
    'positive rows': int(y.sum()),
    'negative rows': int((y == 0).sum()),
    'positive rate': y.mean(),
    'majority-class accuracy': (y == 0).mean(),
}, name='value')
dataset_summary.to_frame()

,value
rows,307511.000000
raw candidate features (including ID before removal),121.000000
positive rows,24825.000000
negative rows,282686.000000
positive rate,0.080729
majority-class accuracy,0.919271


### Fold-local preprocessing design

`src/modeling.py` wraps the completed application preparation in scikit-learn transformers. For every model and search fold, the pipeline runs in this order:

1. learn preparation parameters from the training fold and transform train/validation separately;
2. remove `SK_ID_CURR`;
3. learn numeric median imputation (plus scaling for logistic regression) from the training fold;
4. learn categorical imputation and one-hot categories from the training fold; and
5. fit the classifier.

An audit transformer records the indices it receives during `fit`. Evaluation raises an error if they differ from the expected training indices or overlap the validation indices.

## Modeling process

All models use the same reproducible three-fold stratified splits. The sequence starts with a majority-class reference, compares otherwise identical unweighted and class-balanced logistic regressions, evaluates bounded tree candidates, and then tunes the leading unweighted histogram booster. Learned preparation remains inside each fitted pipeline.

In [3]:
models = build_baseline_models(random_state=DEFAULT_RANDOM_STATE)
list(models)

['Majority-class dummy',
 'Logistic regression (unweighted)',
 'Logistic regression (balanced)']

### Majority-class baseline

The dummy classifier always predicts the most frequent class. Its accuracy is expected to look high because only about 8% of rows are positive, while its AUC must be 0.5 because it cannot rank applicants.

In [4]:
dummy_name = 'Majority-class dummy'
dummy_result = evaluate_model_cv(
    dummy_name,
    models[dummy_name],
    X,
    y,
    n_splits=DEFAULT_N_SPLITS,
    random_state=DEFAULT_RANDOM_STATE,
    splits=cv_splits,
)
dummy_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.0,0.919271,0.5,0.002325,0.000102,0.002427
1,2,205007,102504,0.080729,0.080729,0.0,0.919271,0.5,0.002271,0.000671,0.002942
2,3,205008,102503,0.080729,0.080729,0.0,0.919271,0.5,0.002250,0.000526,0.002777


### Logistic regression: unweighted

This interpretable linear baseline uses ordinary L2-regularized logistic loss.

In [5]:
unweighted_name = 'Logistic regression (unweighted)'
unweighted_result = evaluate_model_cv(
    unweighted_name,
    models[unweighted_name],
    X,
    y,
    n_splits=DEFAULT_N_SPLITS,
    random_state=DEFAULT_RANDOM_STATE,
    splits=cv_splits,
)
unweighted_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.002097,0.919281,0.747959,7.515608,0.824779,8.340387
1,2,205007,102504,0.080729,0.080729,0.001854,0.919154,0.753152,7.323290,0.791670,8.114960
2,3,205008,102503,0.080729,0.080729,0.002176,0.919358,0.748220,24.424726,0.809663,25.234389


### Logistic regression: balanced class weights

The only classifier change is `class_weight='balanced'`. Keeping all other settings and folds identical isolates the effect of weighting.

In [6]:
balanced_name = 'Logistic regression (balanced)'
balanced_result = evaluate_model_cv(
    balanced_name,
    models[balanced_name],
    X,
    y,
    n_splits=DEFAULT_N_SPLITS,
    random_state=DEFAULT_RANDOM_STATE,
    splits=cv_splits,
)
balanced_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.339938,0.688832,0.748230,74.168527,0.780559,74.949087
1,2,205007,102504,0.080729,0.080729,0.343899,0.686822,0.753830,59.409608,0.812300,60.221908
2,3,205008,102503,0.080729,0.080729,0.340175,0.688575,0.747918,91.926837,0.812900,92.739737


## Performance comparisons

### Baseline comparison

In [7]:
results = {
    dummy_name: dummy_result,
    unweighted_name: unweighted_result,
    balanced_name: balanced_result,
}
comparison = (
    pd.DataFrame([result.summary() for result in results.values()])
    .sort_values('mean_roc_auc', ascending=False)
    .reset_index(drop=True)
)
comparison.style.format({
    'mean_accuracy': '{:.6f}',
    'std_accuracy': '{:.6f}',
    'mean_roc_auc': '{:.6f}',
    'std_roc_auc': '{:.6f}',
    'oof_roc_auc': '{:.6f}',
    'mean_fit_seconds': '{:.2f}',
    'total_cv_seconds': '{:.2f}',
})

,model,mean_accuracy,std_accuracy,mean_roc_auc,std_roc_auc,oof_roc_auc,mean_fit_seconds,total_cv_seconds,fold_local_preprocessing_verified
0,Logistic regression (balanced),0.688076,0.001094,0.749993,0.003327,0.749975,75.17,227.91,True
1,Logistic regression (unweighted),0.919265,0.000103,0.749777,0.002926,0.749762,13.09,41.69,True
2,Majority-class dummy,0.919271,0.000000,0.500000,0.000000,0.500000,0.00,0.01,True


### Leakage audit

In [8]:
preprocessing_audit = pd.concat(
    [
        result.preprocessing_audit.assign(model=name)
        for name, result in results.items()
    ],
    ignore_index=True,
).loc[:, [
    'model', 'fold', 'application_fit_rows', 'preprocessor_fit_rows',
    'validation_rows_seen_during_fit', 'verified', 'note'
]]
preprocessing_audit

,model,fold,application_fit_rows,preprocessor_fit_rows,validation_rows_seen_during_fit,verified,note
0,Majority-class dummy,1,NaN,NaN,NaN,True,No feature preprocessing is used by this estim...
1,Majority-class dummy,2,NaN,NaN,NaN,True,No feature preprocessing is used by this estim...
2,Majority-class dummy,3,NaN,NaN,NaN,True,No feature preprocessing is used by this estim...
3,Logistic regression (unweighted),1,205007.0,205007.0,0.0,True,All learned preprocessing was fitted on this f...
4,Logistic regression (unweighted),2,205007.0,205007.0,0.0,True,All learned preprocessing was fitted on this f...
5,Logistic regression (unweighted),3,205008.0,205008.0,0.0,True,All learned preprocessing was fitted on this f...
6,Logistic regression (balanced),1,205007.0,205007.0,0.0,True,All learned preprocessing was fitted on this f...
7,Logistic regression (balanced),2,205007.0,205007.0,0.0,True,All learned preprocessing was fitted on this f...
8,Logistic regression (balanced),3,205008.0,205008.0,0.0,True,All learned preprocessing was fitted on this f...


In [9]:
dummy_summary = dummy_result.summary()
unweighted_summary = unweighted_result.summary()
balanced_summary = balanced_result.summary()
weighting_delta = (
    balanced_summary['mean_roc_auc'] - unweighted_summary['mean_roc_auc']
)
logistic_fold_deltas = (
    balanced_result.fold_metrics['roc_auc']
    - unweighted_result.fold_metrics['roc_auc']
)
display(Markdown(
    f"""### Interpretation

The majority classifier reaches **{dummy_summary['mean_accuracy']:.4f} accuracy** but exactly 
**{dummy_summary['mean_roc_auc']:.3f} ROC-AUC**, confirming that accuracy alone rewards the 
dominant class without providing ranking skill.

The weighted and unweighted logistic models are **essentially tied**. Balanced weighting 
changes mean AUC by only **{weighting_delta:+.6f}**; its fold-level changes are 
**{', '.join(f'{value:+.6f}' for value in logistic_fold_deltas)}**, so the sign is not 
consistent across all folds and the difference is much smaller than either model's fold 
standard deviation. Weighting can still substantially change the number of positive 
classifications at a 0.5 threshold even when ranking AUC moves little.

All logistic folds passed the index-level audit: preparation and downstream preprocessing 
were fitted on the fold's training rows, and **zero validation rows** were seen during fit.
"""
))

### Interpretation

The majority classifier reaches **0.9193 accuracy** but exactly 
**0.500 ROC-AUC**, confirming that accuracy alone rewards the 
dominant class without providing ranking skill.

The weighted and unweighted logistic models are **essentially tied**. Balanced weighting 
changes mean AUC by only **+0.000216**; its fold-level changes are 
**+0.000271, +0.000678, -0.000302**, so the sign is not 
consistent across all folds and the difference is much smaller than either model's fold 
standard deviation. Weighting can still substantially change the number of positive 
classifications at a 0.5 threshold even when ranking AUC moves little.

All logistic folds passed the index-level audit: preparation and downstream preprocessing 
were fitted on the fold's training rows, and **zero validation rows** were seen during fit.


### Candidate-model comparison

This milestone adds one bounded random forest and the requested boosting comparison. The random forest uses 200 trees, maximum depth 16, minimum leaf size 20, 80% bootstrap samples, square-root feature sampling, and balanced subsample weights. Those bounds limit runtime and reduce the tendency of an unrestricted forest to memorize rare observations.

The boosting backend is selected below. Both boosting runs use identical features, folds, preparation, early stopping, and hyperparameters; class weighting is the only difference. The early-stopping validation subset is drawn solely from each outer training fold.


In [10]:
candidate_models, boosting_backend = build_candidate_models(
    random_state=DEFAULT_RANDOM_STATE
)
display(Markdown(f'**Boosting backend:** {boosting_backend}'))
list(candidate_models)

**Boosting backend:** LightGBM is not installed. scikit-learn's HistGradientBoostingClassifier is used as the available gradient-boosting alternative because it uses histogram-binned trees, supports class weights and early stopping, and scales to this data without an added dependency.

['Random forest (balanced subsample)',
 'Histogram gradient boosting (unweighted)',
 'Histogram gradient boosting (balanced)']

### Random forest

In [11]:
random_forest_name = next(
    name for name in candidate_models if name.startswith('Random forest')
)
random_forest_result = evaluate_model_cv(
    random_forest_name,
    candidate_models[random_forest_name],
    X,
    y,
    splits=cv_splits,
)
random_forest_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.175535,0.815090,0.730735,64.561714,1.264560,65.826275
1,2,205007,102504,0.080729,0.080729,0.182286,0.808593,0.729587,64.553349,1.266654,65.820003
2,3,205008,102503,0.080729,0.080729,0.175683,0.814630,0.733391,64.448119,1.197780,65.645899


### Boosting: unweighted

In [12]:
boosting_unweighted_name = next(
    name for name in candidate_models if 'boost' in name.lower() and 'unweighted' in name.lower()
)
boosting_unweighted_result = evaluate_model_cv(
    boosting_unweighted_name,
    candidate_models[boosting_unweighted_name],
    X,
    y,
    splits=cv_splits,
)
boosting_unweighted_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.002137,0.919457,0.757746,21.883065,1.300331,23.183396
1,2,205007,102504,0.080729,0.080729,0.002634,0.919428,0.759391,20.986942,1.254198,22.241139
2,3,205008,102503,0.080729,0.080729,0.002341,0.919563,0.754768,21.322552,1.190948,22.513499


### Boosting: balanced class weights

In [13]:
boosting_balanced_name = next(
    name for name in candidate_models if 'boost' in name.lower() and 'balanced' in name.lower()
)
boosting_balanced_result = evaluate_model_cv(
    boosting_balanced_name,
    candidate_models[boosting_balanced_name],
    X,
    y,
    splits=cv_splits,
)
boosting_balanced_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.314651,0.711738,0.757232,22.426145,1.273355,23.699500
1,2,205007,102504,0.080729,0.080729,0.319792,0.708353,0.759311,19.508004,1.221534,20.729538
2,3,205008,102503,0.080729,0.080729,0.315044,0.710896,0.755045,21.980581,1.343673,23.324255


### Combined results

In [14]:
candidate_results = {
    random_forest_name: random_forest_result,
    boosting_unweighted_name: boosting_unweighted_result,
    boosting_balanced_name: boosting_balanced_result,
}
all_results = {**results, **candidate_results}
combined_results = (
    pd.DataFrame([result.summary() for result in all_results.values()])
    .loc[:, [
        'model', 'mean_roc_auc', 'std_roc_auc', 'oof_roc_auc',
        'mean_accuracy', 'total_cv_seconds',
        'fold_local_preprocessing_verified',
    ]]
    .sort_values('mean_roc_auc', ascending=False)
    .reset_index(drop=True)
)
combined_results.style.format({
    'mean_roc_auc': '{:.6f}',
    'std_roc_auc': '{:.6f}',
    'oof_roc_auc': '{:.6f}',
    'mean_accuracy': '{:.6f}',
    'total_cv_seconds': '{:.2f}',
})

,model,mean_roc_auc,std_roc_auc,oof_roc_auc,mean_accuracy,total_cv_seconds,fold_local_preprocessing_verified
0,Histogram gradient boosting (unweighted),0.757301,0.002343,0.757289,0.919483,67.94,True
1,Histogram gradient boosting (balanced),0.757196,0.002134,0.757166,0.710329,67.75,True
2,Logistic regression (balanced),0.749993,0.003327,0.749975,0.688076,227.91,True
3,Logistic regression (unweighted),0.749777,0.002926,0.749762,0.919265,41.69,True
4,Random forest (balanced subsample),0.731238,0.001951,0.731228,0.812771,197.29,True
5,Majority-class dummy,0.500000,0.000000,0.500000,0.919271,0.01,True


### Candidate fold results

Fold-level results make it possible to distinguish a repeatable gain from a small mean difference driven by one split.

In [15]:
candidate_fold_results = pd.concat(
    [
        result.fold_metrics.assign(model=name)
        for name, result in candidate_results.items()
    ],
    ignore_index=True,
).loc[:, [
    'model', 'fold', 'accuracy', 'roc_auc', 'predicted_positive_rate',
    'fit_seconds', 'score_seconds', 'fold_seconds',
]]
candidate_fold_results.style.format({
    'accuracy': '{:.6f}',
    'roc_auc': '{:.6f}',
    'predicted_positive_rate': '{:.6f}',
    'fit_seconds': '{:.2f}',
    'score_seconds': '{:.2f}',
    'fold_seconds': '{:.2f}',
})

,model,fold,accuracy,roc_auc,predicted_positive_rate,fit_seconds,score_seconds,fold_seconds
0,Random forest (balanced subsample),1,0.815090,0.730735,0.175535,64.56,1.26,65.83
1,Random forest (balanced subsample),2,0.808593,0.729587,0.182286,64.55,1.27,65.82
2,Random forest (balanced subsample),3,0.814630,0.733391,0.175683,64.45,1.20,65.65
3,Histogram gradient boosting (unweighted),1,0.919457,0.757746,0.002137,21.88,1.30,23.18
4,Histogram gradient boosting (unweighted),2,0.919428,0.759391,0.002634,20.99,1.25,22.24
5,Histogram gradient boosting (unweighted),3,0.919563,0.754768,0.002341,21.32,1.19,22.51
6,Histogram gradient boosting (balanced),1,0.711738,0.757232,0.314651,22.43,1.27,23.70
7,Histogram gradient boosting (balanced),2,0.708353,0.759311,0.319792,19.51,1.22,20.73
8,Histogram gradient boosting (balanced),3,0.710896,0.755045,0.315044,21.98,1.34,23.32


### Candidate preprocessing audit

In [16]:
candidate_preprocessing_audit = pd.concat(
    [
        result.preprocessing_audit.assign(model=name)
        for name, result in candidate_results.items()
    ],
    ignore_index=True,
).loc[:, [
    'model', 'fold', 'application_fit_rows', 'preprocessor_fit_rows',
    'validation_rows_seen_during_fit', 'verified',
]]
candidate_preprocessing_audit

,model,fold,application_fit_rows,preprocessor_fit_rows,validation_rows_seen_during_fit,verified
0,Random forest (balanced subsample),1,205007,205007,0,True
1,Random forest (balanced subsample),2,205007,205007,0,True
2,Random forest (balanced subsample),3,205008,205008,0,True
3,Histogram gradient boosting (unweighted),1,205007,205007,0,True
4,Histogram gradient boosting (unweighted),2,205007,205007,0,True
5,Histogram gradient boosting (unweighted),3,205008,205008,0,True
6,Histogram gradient boosting (balanced),1,205007,205007,0,True
7,Histogram gradient boosting (balanced),2,205007,205007,0,True
8,Histogram gradient boosting (balanced),3,205008,205008,0,True


In [17]:
boosting_auc_deltas = (
    boosting_balanced_result.fold_metrics['roc_auc']
    - boosting_unweighted_result.fold_metrics['roc_auc']
)
boosting_mean_delta = boosting_auc_deltas.mean()
best_candidate = max(
    [result.summary() for result in candidate_results.values()],
    key=lambda row: row['mean_roc_auc'],
)
display(Markdown(
    f"""### Candidate interpretation

The strongest initial tree candidate is **{best_candidate['model']}**, with mean CV AUC 
**{best_candidate['mean_roc_auc']:.6f}** and OOF AUC **{best_candidate['oof_roc_auc']:.6f}**.

For boosting, balanced weighting changes mean AUC by **{boosting_mean_delta:+.6f}**. Its 
fold-level changes are **{', '.join(f'{value:+.6f}' for value in boosting_auc_deltas)}**. 
The weighted and unweighted models otherwise use the same data, folds, features, and 
hyperparameters.

The logistic models remain **essentially tied**: their mean difference is only 
**{weighting_delta:+.6f}**, the fold-level sign is mixed, and the difference is small 
relative to their fold variation.

Every candidate fold passed the preprocessing audit with zero outer-validation rows seen 
during fitting.
"""
))

### Candidate interpretation

The strongest initial tree candidate is **Histogram gradient boosting (unweighted)**, with mean CV AUC 
**0.757301** and OOF AUC **0.757289**.

For boosting, balanced weighting changes mean AUC by **-0.000105**. Its 
fold-level changes are **-0.000513, -0.000079, +0.000277**. 
The weighted and unweighted models otherwise use the same data, folds, features, and 
hyperparameters.

The logistic models remain **essentially tied**: their mean difference is only 
**+0.000216**, the fold-level sign is mixed, and the difference is small 
relative to their fold variation.

Every candidate fold passed the preprocessing audit with zero outer-validation rows seen 
during fitting.


## Hyperparameter tuning

### Unweighted histogram boosting

The unweighted histogram booster is tuned with `RandomizedSearchCV` on a reproducible 5,000-row stratified sample. The search runs 20 parameter combinations under 3-fold stratified CV with ROC-AUC scoring. The complete preparation/preprocessing/model pipeline is the searched estimator, so learned transformations are fitted within each search fold. Early stopping uses only an internal subset of that fold's training rows.

The sample search score is used only to select parameters; it is not reported as the full-data performance estimate. Before running the search, the retention rule is fixed: adopt the tuned configuration only if its full-data mean and OOF AUC each improve by at least 0.001, it wins at least two of three paired folds, and no fold falls by more than 0.0005. Otherwise retain the default configuration.

In [18]:
TUNING_SAMPLE_SIZE = 5_000
TUNING_ITERATIONS = 20
MINIMUM_MEAN_AUC_GAIN = 0.001
MINIMUM_FOLD_WINS = 2
MINIMUM_OOF_AUC_GAIN = 0.001
MAXIMUM_FOLD_AUC_REGRESSION = 0.0005
search_space = histogram_boosting_search_space()
possible_combinations = sum(
    int(np.prod([len(values) for values in branch.values()]))
    for branch in search_space
)
search_settings = pd.Series({
    'sample rows': TUNING_SAMPLE_SIZE,
    'randomized combinations': TUNING_ITERATIONS,
    'sample CV folds': DEFAULT_N_SPLITS,
    'scoring': 'roc_auc',
    'random_state': DEFAULT_RANDOM_STATE,
    'parallel search jobs': 1,
    'CV fits plus final sample refit': TUNING_ITERATIONS * DEFAULT_N_SPLITS + 1,
    'possible discrete combinations': possible_combinations,
    'minimum full-data mean AUC gain': MINIMUM_MEAN_AUC_GAIN,
    'minimum paired fold wins': MINIMUM_FOLD_WINS,
    'minimum full-data OOF AUC gain': MINIMUM_OOF_AUC_GAIN,
    'maximum allowed single-fold regression': MAXIMUM_FOLD_AUC_REGRESSION,
}, name='value')
display(search_settings.to_frame())
search_space_table = pd.DataFrame([
    {
        'branch': branch_number,
        'parameter': parameter.removeprefix('classifier__'),
        'candidate values': values,
    }
    for branch_number, branch in enumerate(search_space, start=1)
    for parameter, values in branch.items()
])
search_space_table

,value
sample rows,5000
randomized combinations,20
sample CV folds,3
scoring,roc_auc
random_state,42
parallel search jobs,1
CV fits plus final sample refit,61
possible discrete combinations,2800
minimum full-data mean AUC gain,0.001
minimum paired fold wins,2


,branch,parameter,candidate values
0,1,learning_rate,"[0.025, 0.04, 0.05, 0.075, 0.1]"
1,1,max_iter,"[100, 150, 200, 250]"
2,1,min_samples_leaf,"[20, 40, 60, 100]"
3,1,l2_regularization,"[0.0, 0.1, 1.0, 5.0, 10.0]"
4,1,max_depth,[None]
5,1,max_leaf_nodes,"[15, 31, 63]"
6,2,learning_rate,"[0.025, 0.04, 0.05, 0.075, 0.1]"
7,2,max_iter,"[100, 150, 200, 250]"
8,2,min_samples_leaf,"[20, 40, 60, 100]"
9,2,l2_regularization,"[0.0, 0.1, 1.0, 5.0, 10.0]"


In [19]:
tuning_result = tune_hist_gradient_boosting(
    X,
    y,
    sample_size=TUNING_SAMPLE_SIZE,
    n_iter=TUNING_ITERATIONS,
    n_splits=DEFAULT_N_SPLITS,
    random_state=DEFAULT_RANDOM_STATE,
)
tuning_sample_target = y.loc[tuning_result.sample_indices]
tuning_sample_summary = pd.Series({
    'sample rows': len(tuning_result.sample_indices),
    'sample positive rows': int(tuning_sample_target.sum()),
    'sample positive rate': tuning_result.sample_positive_rate,
    'full-data positive rate': y.mean(),
    'completed combinations': len(tuning_result.ranking),
    'total search seconds (includes final sample refit)': tuning_result.search_seconds,
}, name='value')
tuning_sample_summary.to_frame()

,value
sample rows,5000.000000
sample positive rows,404.000000
sample positive rate,0.080800
full-data positive rate,0.080729
completed combinations,20.000000
total search seconds (includes final sample refit),33.733602


### Sample-only search results

These scores come from the 5,000-row tuning sample and are intentionally labeled separately from the later full-data evaluation.

In [20]:
tuning_result.ranking.head(10).style.format({
    'sample_mean_cv_auc': '{:.6f}',
    'sample_cv_auc_std_ddof0': '{:.6f}',
    'sample_fold_1_auc': '{:.6f}',
    'sample_fold_2_auc': '{:.6f}',
    'sample_fold_3_auc': '{:.6f}',
    'sample_mean_fit_seconds': '{:.3f}',
    'sample_fit_seconds_std': '{:.3f}',
    'learning_rate': '{:.3f}',
    'l2_regularization': '{:.3f}',
})

,sample_rank,sample_mean_cv_auc,sample_cv_auc_std_ddof0,sample_mean_fit_seconds,sample_fit_seconds_std,sample_fold_1_auc,sample_fold_2_auc,sample_fold_3_auc,min_samples_leaf,max_leaf_nodes,max_iter,max_depth,learning_rate,l2_regularization
0,1,0.720072,0.003468,0.501,0.066,0.717078,0.718207,0.724933,60,15,200,None,0.075,10.000
1,2,0.719590,0.010175,0.291,0.006,0.707463,0.718946,0.732361,60,None,200,4,0.100,0.000
2,3,0.718603,0.001704,0.313,0.009,0.717585,0.717220,0.721004,40,None,100,4,0.100,5.000
3,4,0.713936,0.010485,0.511,0.184,0.699200,0.719872,0.722736,100,15,250,None,0.050,10.000
4,5,0.711975,0.004642,0.354,0.013,0.707816,0.709656,0.718454,100,None,150,6,0.100,1.000
5,6,0.711268,0.017189,0.293,0.021,0.687194,0.726226,0.720383,100,None,250,3,0.025,0.000
6,7,0.710851,0.011532,0.330,0.035,0.696335,0.724545,0.711673,60,None,250,4,0.040,0.000
7,8,0.710746,0.002732,0.395,0.055,0.713485,0.707016,0.711737,100,None,150,6,0.075,5.000
8,9,0.710257,0.003318,0.414,0.018,0.712552,0.705565,0.712652,100,31,250,None,0.100,5.000
9,10,0.708329,0.011384,0.610,0.075,0.692895,0.712078,0.720013,20,None,150,6,0.050,5.000


In [21]:
best_sample_params = tuning_result.search.best_params_
selected_parameter_table = pd.Series(
    {
        key.removeprefix('classifier__'): value
        for key, value in best_sample_params.items()
    },
    name='selected value',
).sort_index()
display(Markdown(
    f"**Best sample CV AUC:** `{tuning_result.search.best_score_:.6f}` "
    f"(ranked over {TUNING_ITERATIONS} randomized combinations)."
))
selected_parameter_table.to_frame()

**Best sample CV AUC:** `0.720072` (ranked over 20 randomized combinations).

,selected value
l2_regularization,10.000
learning_rate,0.075
max_depth,NaN
max_iter,200.000
max_leaf_nodes,15.000
min_samples_leaf,60.000


### Full-data evaluation of the sample-selected configuration

The selected parameters are now applied to a fresh pipeline and evaluated on all 307,511 labeled rows using the exact original `cv_splits`. This is the score used for the default-versus-tuned decision. In the parameter tables, pandas renders Python `None` as `NaN`; for the selected model, `max_depth=None` means depth is unrestricted while `max_leaf_nodes=15` provides the structural bound.

In [22]:
tuned_boosting_name = 'Histogram gradient boosting (sample-tuned, unweighted)'
tuned_boosting_pipeline = build_tuned_hist_gradient_boosting_pipeline(
    best_sample_params, random_state=DEFAULT_RANDOM_STATE
)
tuned_boosting_result = evaluate_model_cv(
    tuned_boosting_name,
    tuned_boosting_pipeline,
    X,
    y,
    splits=cv_splits,
)
tuned_boosting_result.fold_metrics

,fold,train_rows,validation_rows,train_positive_rate,validation_positive_rate,predicted_positive_rate,accuracy,roc_auc,fit_seconds,score_seconds,fold_seconds
0,1,205007,102504,0.080729,0.080729,0.002078,0.919398,0.758776,17.564379,1.101119,18.665498
1,2,205007,102504,0.080729,0.080729,0.002732,0.919467,0.759662,17.099039,1.080012,18.179051
2,3,205008,102503,0.080729,0.080729,0.002478,0.919583,0.755380,17.291334,1.106391,18.397724


### Full-data tuned preprocessing audit

In [23]:
tuned_boosting_result.preprocessing_audit

,fold,application_fit_rows,preprocessor_fit_rows,validation_rows_seen_during_fit,verified,note
0,1,205007,205007,0,True,All learned preprocessing was fitted on this f...
1,2,205007,205007,0,True,All learned preprocessing was fitted on this f...
2,3,205008,205008,0,True,All learned preprocessing was fitted on this f...


## Default versus tuned boosting: full-data results

In [24]:
default_tuned_results = (
    pd.DataFrame([
        boosting_unweighted_result.summary(),
        tuned_boosting_result.summary(),
    ])
    .loc[:, [
        'model', 'mean_roc_auc', 'std_roc_auc', 'oof_roc_auc',
        'mean_accuracy', 'total_cv_seconds',
        'fold_local_preprocessing_verified',
    ]]
    .sort_values('mean_roc_auc', ascending=False)
    .reset_index(drop=True)
)
default_tuned_results.style.format({
    'mean_roc_auc': '{:.6f}',
    'std_roc_auc': '{:.6f}',
    'oof_roc_auc': '{:.6f}',
    'mean_accuracy': '{:.6f}',
    'total_cv_seconds': '{:.2f}',
})

,model,mean_roc_auc,std_roc_auc,oof_roc_auc,mean_accuracy,total_cv_seconds,fold_local_preprocessing_verified
0,"Histogram gradient boosting (sample-tuned, unweighted)",0.757940,0.002260,0.757936,0.919483,55.24,True
1,Histogram gradient boosting (unweighted),0.757301,0.002343,0.757289,0.919483,67.94,True


In [25]:
paired_tuning_results, tuning_decision = compare_paired_auc_results(
    boosting_unweighted_result,
    tuned_boosting_result,
    minimum_mean_gain=MINIMUM_MEAN_AUC_GAIN,
    minimum_fold_wins=MINIMUM_FOLD_WINS,
    minimum_oof_gain=MINIMUM_OOF_AUC_GAIN,
    maximum_fold_regression=MAXIMUM_FOLD_AUC_REGRESSION,
)
display(paired_tuning_results.style.format({
    'default_roc_auc': '{:.6f}',
    'tuned_roc_auc': '{:.6f}',
    'tuned_minus_default': '{:+.6f}',
}))
pd.Series(tuning_decision, name='value').to_frame()

,fold,train_rows,validation_rows,default_roc_auc,tuned_roc_auc,tuned_minus_default
0,1,205007,102504,0.757746,0.758776,+0.001031
1,2,205007,102504,0.759391,0.759662,+0.000272
2,3,205008,102503,0.754768,0.755380,+0.000613


,value
mean_auc_gain,0.000638
fold_wins,3
oof_auc_gain,0.000646
worst_fold_auc_change,0.000272
minimum_mean_gain,0.001
minimum_fold_wins,2
minimum_oof_gain,0.001
maximum_fold_regression,0.0005
convincing_improvement,False
selected_configuration,default


In [26]:
default_full_summary = boosting_unweighted_result.summary()
tuned_full_summary = tuned_boosting_result.summary()
selected_boosting_configuration = tuning_decision['selected_configuration']
decision_text = (
    'The tuned configuration meets the predeclared rule and is retained.'
    if tuning_decision['convincing_improvement']
    else 'The tuned configuration does not meet the predeclared rule; the default is retained.'
)
display(Markdown(
    f"""### Tuning decision

The randomized search selected parameters using a **sample CV AUC of 
{tuning_result.search.best_score_:.6f}**. On the full original folds, the default model has 
mean AUC **{default_full_summary['mean_roc_auc']:.6f}** and OOF AUC 
**{default_full_summary['oof_roc_auc']:.6f}**; the tuned model has mean AUC 
**{tuned_full_summary['mean_roc_auc']:.6f}** and OOF AUC 
**{tuned_full_summary['oof_roc_auc']:.6f}**.

The paired mean change is **{tuning_decision['mean_auc_gain']:+.6f}**, the tuned model wins 
**{tuning_decision['fold_wins']} of 3 folds**, and the OOF change is 
**{tuning_decision['oof_auc_gain']:+.6f}**. The worst paired fold change is 
**{tuning_decision['worst_fold_auc_change']:+.6f}**. {decision_text}

Selected configuration for the next stage: **{selected_boosting_configuration}**.
"""
))

### Tuning decision

The randomized search selected parameters using a **sample CV AUC of 
0.720072**. On the full original folds, the default model has 
mean AUC **0.757301** and OOF AUC 
**0.757289**; the tuned model has mean AUC 
**0.757940** and OOF AUC 
**0.757936**.

The paired mean change is **+0.000638**, the tuned model wins 
**3 of 3 folds**, and the OOF change is 
**+0.000646**. The worst paired fold change is 
**+0.000272**. The tuned configuration does not meet the predeclared rule; the default is retained.

Selected configuration for the next stage: **default**.


### Tuning limitations

- The 5,000-row sample contains relatively few positive cases, and each search fold's internal early-stopping subset is smaller still; sample rankings are therefore noisy.
- Twenty randomized combinations cover only a small fraction of the bounded discrete search space. `max_iter` is an upper bound because early stopping may halt earlier. The sample CV standard deviation shown by scikit-learn uses `ddof=0`; the full-data fold SD uses `ddof=1`.
- The tuning sample is drawn from the full labeled data, so the later full-fold comparison is not a strictly nested-CV estimate: some observations helped select parameters before appearing in a full validation fold. The requested full-data comparison is useful for a practical decision but may be mildly optimistic. A fully nested search would be more expensive.
- With only three paired folds, the retention rule is a practical repeatability threshold, not a formal significance test. Search and CV runtime are hardware- and session-dependent.
- The next section fits the retained default configuration. Blending, SHAP, bootstrap intervals, and subgroup analysis remain deferred.

## Final model selection

### Final fit and Kaggle submission

The retained model is the **default, unweighted histogram gradient booster**. The sample-tuned model's full-data gains (`+0.000638` mean fold AUC and `+0.000646` OOF AUC) were positive but below the predeclared practical thresholds of `+0.001`, so the simpler prespecified configuration is used.

Final fitting uses two training-only stages. First, the retained pipeline is fitted to the full labeled table with its original internal 10% early-stopping validation split, ROC-AUC scoring, 15-round patience, and 200-iteration cap. Its observed `n_iter_` is recorded. Second, a fresh copy of the same pipeline disables early stopping, fixes `max_iter` to that observed count, and refits on every labeled row. This lets the saved production classifier use all labeled rows without consulting the competition test set. The observed stop count can include patience rounds, so it is reported separately from the best internal validation-score iteration.

Application preparation and encoded-feature preprocessing are fitted on the full labeled table before the classifier's internal calibration split; no competition-test row and no test-derived statistic is used during fitting. `TARGET` is separated before the pipeline, and `SK_ID_CURR` is removed by the application-preparation step before predictors reach the model.

In [27]:
TEST_PATH = PROJECT_ROOT / 'data/raw/application_test.csv'
SAMPLE_SUBMISSION_PATH = PROJECT_ROOT / 'data/raw/sample_submission.csv'
MODEL_PATH = PROJECT_ROOT / 'models/final_hist_gradient_boosting_pipeline.joblib'
CONFIG_PATH = PROJECT_ROOT / 'models/final_hist_gradient_boosting_config.json'
SUBMISSION_PATH = PROJECT_ROOT / 'submissions/home_credit_hist_gradient_boosting.csv'

test = load_test_data(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
if list(test.columns) != list(X.columns):
    raise ValueError('Raw training and test predictor schemas or column order differ.')
if not sample_submission['SK_ID_CURR'].equals(test['SK_ID_CURR']):
    raise ValueError('Sample-submission IDs do not match test IDs in row order.')

final_input_summary = pd.Series({
    'labeled training rows': len(X),
    'competition test rows': len(test),
    'raw columns entering pipeline (including ID)': X.shape[1],
    'unique test IDs': test['SK_ID_CURR'].nunique(),
    'test contains TARGET': 'TARGET' in test.columns,
    'sample IDs match test in order': sample_submission['SK_ID_CURR'].equals(test['SK_ID_CURR']),
}, name='value')
final_input_summary.to_frame()

,value
labeled training rows,307511
competition test rows,48744
raw columns entering pipeline (including ID),121
unique test IDs,48744
test contains TARGET,False
sample IDs match test in order,True


In [28]:
final_fit = fit_final_hist_gradient_boosting(
    X, y, random_state=DEFAULT_RANDOM_STATE
)
final_application = final_fit.pipeline.named_steps['application_preparation']
final_encoder = final_fit.pipeline.named_steps['feature_preprocessor']
final_classifier = final_fit.pipeline.named_steps['classifier']

final_fit_summary = pd.Series({
    'model': 'Default unweighted histogram gradient boosting',
    'calibration iteration cap': final_fit.calibration_iteration_cap,
    'calibration stopped iteration count': final_fit.stopped_iteration_count,
    'calibration stopped before cap': final_fit.stopped_early,
    'best internal validation-score iteration': final_fit.best_validation_iteration,
    'best internal validation ROC-AUC': final_fit.best_validation_score,
    'calibration fit seconds': final_fit.calibration_fit_seconds,
    'final fixed-iteration fit seconds': final_fit.final_fit_seconds,
    'final preprocessing fit rows': final_application.fit_row_count_,
    'final classifier iterations': final_classifier.n_iter_,
    'final classifier early_stopping': final_classifier.early_stopping,
    'final encoded predictor count': len(final_encoder.get_feature_names_out()),
    'SK_ID_CURR in prepared predictors': 'SK_ID_CURR' in set(final_application.get_feature_names_out()),
    'TARGET in raw pipeline input': 'TARGET' in set(final_application.feature_names_in_),
}, name='value')
final_fit_summary.to_frame()

,value
model,Default unweighted histogram gradient boosting
calibration iteration cap,200
calibration stopped iteration count,200
calibration stopped before cap,False
best internal validation-score iteration,199
best internal validation ROC-AUC,0.767595
calibration fit seconds,31.335468
final fixed-iteration fit seconds,32.659581
final preprocessing fit rows,307511
final classifier iterations,200


### Iteration handling

`n_iter_` is the number of boosting iterations scikit-learn actually fitted before stopping, not necessarily the iteration with the single highest internal validation score; it may include the 15 no-improvement patience rounds. The second-stage production refit deliberately uses that observed count because it mirrors the stopping behavior of the retained CV estimator, but it is not identical to the CV training protocol: the final classifier has no internal holdout and sees every labeled row. The reported CV AUC values remain the performance evidence; the internal full-data calibration score is used only to choose the iteration count.

In [29]:
submission, submission_validation = create_kaggle_submission(
    final_fit.pipeline, test, sample_submission, SUBMISSION_PATH
)
KAGGLE_PUBLIC_AUC = 0.74531
KAGGLE_PRIVATE_AUC = 0.74479

final_configuration = {
    'artifact_schema_version': 1,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_choice': 'default unweighted histogram gradient boosting',
    'selection_reason': (
        'The sample-tuned configuration improved full-data mean CV AUC by '
        f"{tuning_decision['mean_auc_gain']:+.6f} and OOF AUC by "
        f"{tuning_decision['oof_auc_gain']:+.6f}; both gains were below "
        'the predeclared +0.001 practical thresholds.'
    ),
    'random_state': DEFAULT_RANDOM_STATE,
    'target_column': 'TARGET',
    'id_column': 'SK_ID_CURR',
    'excluded_from_predictors': ['TARGET', 'SK_ID_CURR'],
    'training_data': {
        'path': str(TRAIN_PATH),
        'rows': len(X),
        'positive_rows': int(y.sum()),
        'positive_rate': float(y.mean()),
        'raw_pipeline_input_columns': X.shape[1],
        'encoded_predictor_count': len(final_encoder.get_feature_names_out()),
    },
    'software': {
        'python': platform.python_version(),
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit_learn': sklearn.__version__,
    },
    'classifier_parameters': final_classifier.get_params(deep=False),
    'class_labels': final_classifier.classes_.tolist(),
    'preparation_parameters': final_application.parameters_.to_dict(),
    'early_stopping_calibration': {
        'uses_competition_test_data': False,
        'scoring': 'roc_auc',
        'validation_fraction': 0.1,
        'n_iter_no_change': 15,
        'iteration_cap': final_fit.calibration_iteration_cap,
        'stopped_iteration_count': final_fit.stopped_iteration_count,
        'stopped_before_cap': final_fit.stopped_early,
        'best_validation_score_iteration': final_fit.best_validation_iteration,
        'best_validation_roc_auc': final_fit.best_validation_score,
        'fit_seconds': final_fit.calibration_fit_seconds,
    },
    'final_refit': {
        'early_stopping': False,
        'validation_fraction': None,
        'max_iter': final_fit.stopped_iteration_count,
        'rows': len(X),
        'fit_seconds': final_fit.final_fit_seconds,
    },
    'selection_evidence': {
        'baseline_and_candidate_results': combined_results.to_dict(orient='records'),
        'full_data_default_vs_tuned': default_tuned_results.to_dict(orient='records'),
        'paired_fold_differences': paired_tuning_results.to_dict(orient='records'),
        'decision': tuning_decision,
    },
    'submission_validation': submission_validation,
    'kaggle_score': {
        'status': 'succeeded',
        'metric': 'roc_auc',
        'public_auc': KAGGLE_PUBLIC_AUC,
        'private_auc': KAGGLE_PRIVATE_AUC,
        'submission_file': SUBMISSION_PATH.name,
    },
}
artifact_details = save_pipeline_and_configuration(
    final_fit.pipeline, MODEL_PATH, final_configuration, CONFIG_PATH
)

reloaded_pipeline = load_saved_pipeline(MODEL_PATH)
reloaded_classes = np.asarray(reloaded_pipeline.classes_)
reloaded_positive_index = int(np.flatnonzero(reloaded_classes == 1)[0])
reloaded_probabilities = reloaded_pipeline.predict_proba(test)[:, reloaded_positive_index]
saved_pipeline_predictions_match = np.allclose(
    reloaded_probabilities, submission['TARGET'].to_numpy(), rtol=0.0, atol=0.0
)
if not saved_pipeline_predictions_match:
    raise AssertionError('Reloaded-pipeline predictions differ from the saved submission.')

display_artifact_details = {
    **artifact_details,
    'model_path': str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    'configuration_path': str(CONFIG_PATH.relative_to(PROJECT_ROOT)),
}
display_submission_validation = {
    **submission_validation,
    'submission_path': str(SUBMISSION_PATH.relative_to(PROJECT_ROOT)),
}
artifact_and_validation = pd.Series({
    **display_artifact_details,
    **display_submission_validation,
    'saved_pipeline_predictions_match': saved_pipeline_predictions_match,
    'Kaggle public AUC': KAGGLE_PUBLIC_AUC,
    'Kaggle private AUC': KAGGLE_PRIVATE_AUC,
}, name='value')
artifact_and_validation.to_frame()

,value
model_path,models/final_hist_gradient_boosting_pipeline.joblib
model_bytes,370239
model_sha256,8aae78412c60eb6eced044d81dc7b8651b960b65fdd67a...
configuration_path,models/final_hist_gradient_boosting_config.json
configuration_bytes,9749
configuration_sha256,7666161d34672393e8863bb7850d90652b76c172fb2007...
columns_exact,True
column_order,"[SK_ID_CURR, TARGET]"
row_count,48744
expected_row_count,48744


In [30]:
display(Markdown(
    f"""### Submission handoff

The final CSV is `{SUBMISSION_PATH.relative_to(PROJECT_ROOT)}`. It has **{submission_validation['row_count']:,} rows** and **{submission_validation['unique_id_count']:,} unique IDs**. IDs match both `sample_submission.csv` and `application_test.csv` in exact row order. All predictions are finite and within `[0, 1]` (minimum **{submission_validation['prediction_min']:.6f}**, maximum **{submission_validation['prediction_max']:.6f}**, mean **{submission_validation['prediction_mean']:.6f}**). The written CSV was reread and revalidated within float64 round-trip tolerance; the reloaded saved pipeline exactly reproduces the in-memory probabilities used to write it.

The saved pipeline is `{MODEL_PATH.relative_to(PROJECT_ROOT)}` and its configuration is `{CONFIG_PATH.relative_to(PROJECT_ROOT)}`.

**Authenticated Kaggle AUC:** public **{KAGGLE_PUBLIC_AUC:.5f}**; private **{KAGGLE_PRIVATE_AUC:.5f}**.
"""
))

### Submission handoff

The final CSV is `submissions/home_credit_hist_gradient_boosting.csv`. It has **48,744 rows** and **48,744 unique IDs**. IDs match both `sample_submission.csv` and `application_test.csv` in exact row order. All predictions are finite and within `[0, 1]` (minimum **0.006439**, maximum **0.700170**, mean **0.079486**). The written CSV was reread and revalidated within float64 round-trip tolerance; the reloaded saved pipeline exactly reproduces the in-memory probabilities used to write it.

The saved pipeline is `models/final_hist_gradient_boosting_pipeline.joblib` and its configuration is `models/final_hist_gradient_boosting_config.json`.

**Authenticated Kaggle AUC:** public **0.74531**; private **0.74479**.


## Kaggle results

The authenticated submission of `home_credit_hist_gradient_boosting.csv` succeeded. These leaderboard AUCs are held-out competition results and are reported separately from the local three-fold stratified cross-validation estimate.

| Evaluation source | ROC-AUC | Difference from mean CV AUC |
|---|---:|---:|
| Local mean 3-fold CV | **0.757301** | — |
| Kaggle public leaderboard | **0.74531** | -0.011991 |
| Kaggle private leaderboard | **0.74479** | -0.012511 |

The leaderboard scores are about 0.012 AUC below local CV, so the local estimate was modestly optimistic for the competition test data. Plausible contributors include sampling variation, model-development selection, and train/test distribution differences; the evaluation protocols and populations are not identical. The public and private scores differ by only 0.00052, indicating stable ranking performance across Kaggle's two held-out partitions.

### Final limitations

- Competition labels remain unavailable locally; the recorded public and private AUCs come from Kaggle and cannot be independently recomputed from the project files.
- The fixed-iteration production refit uses all labeled rows and therefore differs slightly from the early-stopped fold models that produced the CV estimates. Its iteration count is a reproducible training-only heuristic, not a new performance estimate.
- Only application-table features are used. The bounded randomized search was not exhaustive, and its 5,000-row selection sample overlaps the population used for the requested full-data comparison.
- Probability calibration was not evaluated. Blending, SHAP, bootstrap intervals, and subgroup analysis remain deferred.